In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score

import lightgbm as lgb
import joblib

from model import DemandModel

In [2]:
df = pd.read_csv("dataset/train.csv", index_col='Index')
df.head()

,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
Index,,,,,,,,,,
0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [4]:
from sklearn.model_selection import train_test_split

tv_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [7]:
print(len(tv_df), len(test_df))

61839 15460


In [8]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.02,

    "num_leaves": 128,
    "max_depth": 8,
    "min_data_in_leaf": 30,

    "feature_fraction": 0.6,
    "bagging_fraction": 0.6,
    "bagging_freq": 5,

    "lambda_l1": 2.0,
    "lambda_l2": 8.0,

    "min_gain_to_split": 0.01,

    "verbosity": -1
}

In [ ]:
from sklearn.model_selection import GroupKFold, KFold

x = tv_df.copy()
y = tv_df["demand"]

kf = KFold(n_splits=4,
           shuffle=True,
           random_state=42)
oof = np.zeros(len(tv_df))

for fold, (tr, va) in enumerate(kf.split(x, y)):

    train_fold = tv_df.iloc[tr]
    val_fold = tv_df.iloc[va]

    model = DemandModel(params)
    model.fit(train_fold, val_fold)

    preds = model.predict(val_fold)
    oof[va] = preds

    print(f"Fold {fold + 1} R2:", r2_score(val_fold["demand"], preds))

print("OOF R2:", r2_score(y, oof))

Training until validation scores don't improve for 100 rounds
[50]	valid_0's rmse: 0.0761822
[100]	valid_0's rmse: 0.0543715
[150]	valid_0's rmse: 0.0487309
[200]	valid_0's rmse: 0.046394
[250]	valid_0's rmse: 0.0455797
[300]	valid_0's rmse: 0.0451738
[350]	valid_0's rmse: 0.0450273
[400]	valid_0's rmse: 0.044893
[450]	valid_0's rmse: 0.044825
[500]	valid_0's rmse: 0.0447646
[550]	valid_0's rmse: 0.0447498
[600]	valid_0's rmse: 0.0447022
[650]	valid_0's rmse: 0.0446757
[700]	valid_0's rmse: 0.0446564
[750]	valid_0's rmse: 0.0446212
[800]	valid_0's rmse: 0.0446051
[850]	valid_0's rmse: 0.0446022
[900]	valid_0's rmse: 0.0446022
Early stopping, best iteration is:
[818]	valid_0's rmse: 0.0446022
Fold 1 R2: 0.9093653639681728
Training until validation scores don't improve for 100 rounds
[50]	valid_0's rmse: 0.0723455
[100]	valid_0's rmse: 0.0515223
[150]	valid_0's rmse: 0.0463399
[200]	valid_0's rmse: 0.0442328
[250]	valid_0's rmse: 0.0435874
[300]	valid_0's rmse: 0.0432573
[350]	valid_0's 

In [10]:
test_y = test_df["demand"]
test_x = test_df.drop(columns=["demand"])
preds = model.predict(test_x)

print("Test R2:", r2_score(test_y, preds))

Test R2: 0.9055843820597157


In [11]:
joblib.dump(model.model, "model.pkl")
joblib.dump(model.pipeline, "pipeline.pkl")

['pipeline.pkl']

### -----TESTING ROUTINE----- ###

In [12]:
submission_test_df = pd.read_csv("dataset/test.csv")
predictions = model.predict(submission_test_df)

submission = pd.DataFrame({
    "Index": submission_test_df.index,
    "demand": predictions
})

submission.to_csv("dataset/submission.csv", index=False)